# TRACE: The Transition Region and Coronal Explorer — Implementation
# TRACE: 천이영역 및 코로나 탐사선 — 구현

**Paper**: Handy, B.N., et al. (1999). "The Transition Region and Coronal Explorer." *Solar Physics*, 187, 229–260. [DOI: 10.1023/A:1005166902804]

---

이 노트북은 TRACE 위성의 주요 특성을 정량적으로 분석하고, EIT 및 후속 기기들과의 비교를 통해 TRACE의 과학적 기여를 시각화합니다.

This notebook quantitatively analyzes the key characteristics of the TRACE satellite, visualizing its scientific contributions through comparison with EIT and subsequent instruments.

**구조 / Structure**:
1. TRACE vs EIT 광학 비교 / Optical Comparison
2. 온도 커버리지 / Temperature Coverage
3. 코로나 루프 분해능 시뮬레이션 / Coronal Loop Resolution Simulation
4. 데이터 볼륨 비교 / Data Volume Comparison
5. EUV 영상 진화 / EUV Imaging Evolution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Arc, FancyArrowPatch
from matplotlib.gridspec import GridSpec
from scipy.ndimage import gaussian_filter

# Global plot settings.
plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 9,
    'figure.facecolor': 'white',
})

---
## Part 1: TRACE vs EIT 광학 비교 / TRACE vs EIT Optical Comparison

TRACE와 EIT의 광학 시스템을 정량적으로 비교합니다. 구경, 초점거리, 화소 크기, 회절 한계 등을 분석합니다.

We quantitatively compare the optical systems of TRACE and EIT, analyzing aperture, focal length, pixel scale, and diffraction limits.

### (a) 분해능 비교 / Resolution Comparison

| Parameter | EIT | TRACE |
|-----------|-----|-------|
| Aperture / 구경 | 12 cm | 30 cm |
| Focal length / 초점거리 | 165 cm | 866 cm |
| Pixel scale / 화소 크기 | 2.6"/pixel | 0.5"/pixel |
| Diffraction limit @ 171Å | ~0.35" | ~0.14" |
| Physical resolution / 물리 분해능 | ~1900 km | ~360 km |

In [ ]:
# =============================================================================
# Part 1(a): PSF comparison — EIT vs TRACE at 171 Angstrom
# =============================================================================

# Instrument parameters.
eit_aperture_cm = 12.0
eit_focal_cm = 165.0
eit_pixel_arcsec = 2.6

trace_aperture_cm = 30.0
trace_focal_cm = 866.0
trace_pixel_arcsec = 0.5

# Diffraction limit: theta = 1.22 * lambda / D (in radians).
wavelength_171_cm = 171e-8  # 171 Angstrom in cm.

def diffraction_limit_arcsec(wavelength_cm: float, aperture_cm: float) -> float:
    """Compute diffraction limit in arcsec using Rayleigh criterion."""
    theta_rad = 1.22 * wavelength_cm / aperture_cm
    return np.degrees(theta_rad) * 3600

eit_diff = diffraction_limit_arcsec(wavelength_171_cm, eit_aperture_cm)
trace_diff = diffraction_limit_arcsec(wavelength_171_cm, trace_aperture_cm)

print(f"EIT  diffraction limit @ 171A: {eit_diff:.3f} arcsec")
print(f"TRACE diffraction limit @ 171A: {trace_diff:.3f} arcsec")

# Build 2D PSF: Gaussian with FWHM = max(diffraction_limit, pixel_scale)
# convolved with a pixel box function.
def make_psf(pixel_arcsec: float, diff_arcsec: float,
             grid_size: float = 10.0, sampling: float = 0.02) -> tuple:
    """Generate a 2D PSF (Gaussian core convolved with pixel response).

    Args:
        pixel_arcsec: Pixel scale in arcsec.
        diff_arcsec: Diffraction limit in arcsec.
        grid_size: Half-size of the grid in arcsec.
        sampling: Grid sampling in arcsec.

    Returns:
        Tuple of (x_axis, y_axis, psf_2d).
    """
    x = np.arange(-grid_size, grid_size + sampling, sampling)
    y = np.arange(-grid_size, grid_size + sampling, sampling)
    xx, yy = np.meshgrid(x, y)

    # Gaussian core (Airy approximation): FWHM ~ diffraction limit.
    sigma = diff_arcsec / (2 * np.sqrt(2 * np.log(2)))
    gaussian = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))

    # Pixel box function.
    pixel_box = np.where(
        (np.abs(xx) <= pixel_arcsec / 2) & (np.abs(yy) <= pixel_arcsec / 2),
        1.0, 0.0
    )
    pixel_box /= pixel_box.sum()

    # Convolve Gaussian with pixel response using FFT.
    from scipy.signal import fftconvolve
    psf = fftconvolve(gaussian, pixel_box, mode='same')
    psf /= psf.max()
    return x, y, psf

x_eit, y_eit, psf_eit = make_psf(eit_pixel_arcsec, eit_diff)
x_trace, y_trace, psf_trace = make_psf(trace_pixel_arcsec, trace_diff)

# Plot PSF comparison.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# 2D PSF — EIT.
im0 = axes[0].imshow(
    psf_eit, extent=[x_eit[0], x_eit[-1], y_eit[0], y_eit[-1]],
    cmap='inferno', origin='lower', vmin=0, vmax=1
)
axes[0].set_title('EIT PSF @ 171 A')
axes[0].set_xlabel('Arcsec')
axes[0].set_ylabel('Arcsec')
axes[0].set_xlim(-5, 5)
axes[0].set_ylim(-5, 5)
fig.colorbar(im0, ax=axes[0], shrink=0.8)

# 2D PSF — TRACE.
im1 = axes[1].imshow(
    psf_trace, extent=[x_trace[0], x_trace[-1], y_trace[0], y_trace[-1]],
    cmap='inferno', origin='lower', vmin=0, vmax=1
)
axes[1].set_title('TRACE PSF @ 171 A')
axes[1].set_xlabel('Arcsec')
axes[1].set_ylabel('Arcsec')
axes[1].set_xlim(-5, 5)
axes[1].set_ylim(-5, 5)
fig.colorbar(im1, ax=axes[1], shrink=0.8)

# 1D radial profiles.
mid_eit = psf_eit.shape[0] // 2
mid_trace = psf_trace.shape[0] // 2
axes[2].plot(x_eit, psf_eit[mid_eit, :], 'r-', lw=2, label='EIT (2.6"/pix)')
axes[2].plot(x_trace, psf_trace[mid_trace, :], 'b-', lw=2, label='TRACE (0.5"/pix)')
axes[2].axhline(0.5, color='gray', ls='--', alpha=0.5, label='FWHM level')
axes[2].set_xlim(-5, 5)
axes[2].set_xlabel('Arcsec')
axes[2].set_ylabel('Normalized Intensity')
axes[2].set_title('Radial PSF Profile Comparison')
axes[2].legend()

fig.suptitle('Part 1(a): PSF Comparison — EIT vs TRACE @ 171 A', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### (b) FOV vs 분해능 트레이드오프 / FOV vs Resolution Tradeoff

EIT는 전체 태양 디스크(45')를 커버하지만 분해능이 2.6"로 제한됩니다. TRACE는 8.5' FOV만 커버하지만 0.5" 분해능으로 미세 구조를 관측합니다.

EIT covers the full solar disk (45') but is limited to 2.6" resolution. TRACE covers only an 8.5' FOV but resolves fine structures at 0.5" resolution.

- 1 arcsec ≈ 725 km on the Sun (at 1 AU)
- TRACE는 ~360 km 규모의 구조를 분해 가능 / TRACE resolves structures down to ~360 km
- EIT는 ~1900 km 규모까지만 분해 / EIT resolves only down to ~1900 km

In [ ]:
# =============================================================================
# Part 1(b): FOV vs Resolution tradeoff — solar disk coverage
# =============================================================================

# Solar disk apparent diameter: ~32 arcmin (1920 arcsec).
solar_radius_arcsec = 960.0  # arcsec

# FOV in arcsec.
eit_fov_arcsec = 45.0 * 60   # 45 arcmin
trace_fov_arcsec = 8.5 * 60  # 8.5 arcmin

# Physical scale: 1 arcsec ~ 725 km.
km_per_arcsec = 725.0

eit_resolution_km = eit_pixel_arcsec * km_per_arcsec
trace_resolution_km = trace_pixel_arcsec * km_per_arcsec

print(f"EIT  pixel resolution: {eit_pixel_arcsec}\" = {eit_resolution_km:.0f} km")
print(f"TRACE pixel resolution: {trace_pixel_arcsec}\" = {trace_resolution_km:.0f} km")
print(f"\
EIT  FOV fraction of disk: {(eit_fov_arcsec / (2 * solar_radius_arcsec)):.1%}")
print(f"TRACE FOV fraction of disk: {(trace_fov_arcsec / (2 * solar_radius_arcsec)):.1%}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: FOV comparison on solar disk.
for ax in axes[:1]:
    # Draw solar disk.
    sun = Circle((0, 0), solar_radius_arcsec, fill=True,
                 facecolor='#FFE0B2', edgecolor='darkorange', lw=2)
    ax.add_patch(sun)

    # EIT FOV (full disk, square).
    eit_half = eit_fov_arcsec / 2
    eit_rect = plt.Rectangle(
        (-eit_half, -eit_half), eit_fov_arcsec, eit_fov_arcsec,
        fill=False, edgecolor='red', lw=2, ls='--', label='EIT FOV (45\')'
    )
    ax.add_patch(eit_rect)

    # TRACE FOV (small square, pointed at active region).
    trace_half = trace_fov_arcsec / 2
    # Place TRACE FOV at a typical active region location.
    trace_cx, trace_cy = 300, 200
    trace_rect = plt.Rectangle(
        (trace_cx - trace_half, trace_cy - trace_half),
        trace_fov_arcsec, trace_fov_arcsec,
        fill=False, edgecolor='blue', lw=2, label='TRACE FOV (8.5\')'
    )
    ax.add_patch(trace_rect)

    ax.set_xlim(-1300, 1300)
    ax.set_ylim(-1300, 1300)
    ax.set_aspect('equal')
    ax.set_xlabel('Arcsec')
    ax.set_ylabel('Arcsec')
    ax.set_title('FOV Coverage on Solar Disk')
    ax.legend(loc='lower left', fontsize=10)
    ax.grid(True, alpha=0.3)

# Right: Resolution vs FOV scatter.
ax2 = axes[1]
instruments = ['EIT', 'TRACE']
fovs = [45.0, 8.5]  # arcmin
resolutions = [eit_pixel_arcsec, trace_pixel_arcsec]  # arcsec
colors = ['red', 'blue']

for name, fov, res, c in zip(instruments, fovs, resolutions, colors):
    ax2.scatter(fov, res, s=200, c=c, zorder=5, edgecolors='black')
    ax2.annotate(
        f'{name}\
{fov}\' / {res}"',
        (fov, res), textcoords='offset points', xytext=(15, 10),
        fontsize=10, color=c, fontweight='bold'
    )

# Add SDO/AIA for context.
ax2.scatter(41.0, 0.6, s=200, c='green', zorder=5,
            edgecolors='black', marker='D')
ax2.annotate(
    'AIA\
41\' / 0.6"', (41.0, 0.6),
    textcoords='offset points', xytext=(15, 10),
    fontsize=10, color='green', fontweight='bold'
)

ax2.set_xlabel('Field of View (arcmin)')
ax2.set_ylabel('Pixel Scale (arcsec)')
ax2.set_title('FOV vs Resolution Tradeoff')
ax2.set_xlim(0, 55)
ax2.set_ylim(0, 3.5)
ax2.grid(True, alpha=0.3)

# Annotate the tradeoff.
ax2.annotate(
    'Ideal: large FOV +\
high resolution',
    xy=(40, 0.3), fontsize=9, style='italic', color='gray',
    ha='center'
)

fig.suptitle('Part 1(b): FOV vs Resolution Tradeoff', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### (c) 다층막 반사율 비교 / Multilayer Reflectivity Comparison (Fig. 7)

EIT는 Mo/Si 다층막을, TRACE는 개선된 Mo₂C/Si 다층막을 사용합니다. TRACE의 다층막은 더 높은 피크 반사율과 더 좁은 대역폭을 제공합니다.

EIT uses Mo/Si multilayers while TRACE uses improved Mo₂C/Si multilayers. TRACE's multilayers provide higher peak reflectivity and narrower bandwidth.

- EIT: 2-bounce reflectivity ~3-7% (Mo/Si)
- TRACE: 2-bounce reflectivity ~3.5-11% (Mo₂C/Si)
- 반사율 향상은 더 높은 감도로 직결 / Higher reflectivity directly translates to better sensitivity

In [ ]:
# =============================================================================
# Part 1(c): Multilayer reflectivity comparison (Fig. 7 style)
# =============================================================================

def gaussian_band(wavelength: np.ndarray, center: float,
                  fwhm: float, peak: float) -> np.ndarray:
    """Model a multilayer reflectivity curve as a Gaussian bandpass.

    Args:
        wavelength: Wavelength array in Angstrom.
        center: Center wavelength in Angstrom.
        fwhm: Full width at half maximum in Angstrom.
        peak: Peak 2-bounce reflectivity (fraction).

    Returns:
        Reflectivity array.
    """
    sigma = fwhm / (2 * np.sqrt(2 * np.log(2)))
    return peak * np.exp(-0.5 * ((wavelength - center) / sigma) ** 2)


wavelength = np.linspace(150, 320, 2000)

# EIT Mo/Si multilayer parameters (approximate from literature).
# 2-bounce reflectivity values.
eit_171 = gaussian_band(wavelength, 171.1, 6.0, 0.030)
eit_195 = gaussian_band(wavelength, 195.1, 6.5, 0.050)
eit_284 = gaussian_band(wavelength, 284.2, 10.0, 0.070)

# TRACE Mo2C/Si multilayer parameters (from paper Fig. 7).
trace_171 = gaussian_band(wavelength, 171.1, 5.0, 0.040)
trace_195 = gaussian_band(wavelength, 195.1, 5.5, 0.110)
trace_284 = gaussian_band(wavelength, 284.2, 8.0, 0.035)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=False)

bands = [
    ('171 A (Fe IX/X)', 171, eit_171, trace_171, (160, 185)),
    ('195 A (Fe XII)', 195, eit_195, trace_195, (182, 210)),
    ('284 A (Fe XV)', 284, eit_284, trace_284, (268, 300)),
]

for ax, (title, center, eit_r, trace_r, xlim) in zip(axes, bands):
    ax.plot(wavelength, eit_r * 100, 'r-', lw=2, label='EIT (Mo/Si)')
    ax.plot(wavelength, trace_r * 100, 'b-', lw=2, label='TRACE (Mo2C/Si)')
    ax.axvline(center, color='gray', ls=':', alpha=0.5)
    ax.set_xlim(xlim)
    ax.set_xlabel('Wavelength (A)')
    ax.set_ylabel('2-Bounce Reflectivity (%)')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle(
    'Part 1(c): Multilayer Reflectivity — EIT (Mo/Si) vs TRACE (Mo2C/Si)',
    fontsize=13, y=1.02
)
plt.tight_layout()
plt.show()

# Print peak reflectivities.
print("\
2-Bounce Peak Reflectivity Comparison:")
print(f"{'Band':>8s}  {'EIT (Mo/Si)':>12s}  {'TRACE (Mo2C/Si)':>16s}  {'Improvement':>12s}")
for name, eit_peak, trace_peak in [
    ('171 A', 3.0, 4.0), ('195 A', 5.0, 11.0), ('284 A', 7.0, 3.5)
]:
    ratio = trace_peak / eit_peak
    print(f"{name:>8s}  {eit_peak:>11.1f}%  {trace_peak:>15.1f}%  {ratio:>11.1f}x")

---
## Part 2: 온도 커버리지 / Temperature Coverage

TRACE의 8개 채널이 커버하는 온도 범위를 시각화합니다. Table I의 데이터를 기반으로 합니다.

Visualize the temperature coverage of TRACE's 8 channels based on Table I data.

### (a) TRACE 채널별 온도 응답 / TRACE Channel Temperature Response

| Channel | Ion | Temperature Range | Peak T |
|---------|-----|-------------------|--------|
| 171 A | Fe IX/X | 1.6-20 x 10^5 K | ~1 MK |
| 195 A | Fe XII/XXIV | 5x10^5 - 2.6x10^7 K | ~1.5 MK |
| 284 A | Fe XV | 1.25-4 x 10^6 K | ~2 MK |
| 1216 A | H I Ly-alpha | 1-3 x 10^4 K | ~2x10^4 K |
| 1550 A | C IV | 6-25 x 10^4 K | ~1x10^5 K |
| 1600 A | UV continuum | 4-10 x 10^3 K | ~5x10^3 K |
| 1700 A | Continuum | 4-10 x 10^3 K | ~5x10^3 K |
| WL | White light | 4-6.4 x 10^3 K | ~5.8x10^3 K |

In [ ]:
# =============================================================================
# Part 2(a): Temperature coverage of TRACE channels
# =============================================================================

# Channel data: (name, log_T_min, log_T_max, log_T_peak, color, ion).
trace_channels = [
    ('WL',      3.60, 3.81, 3.76, '#8B4513',  'Photosphere'),
    ('1700 A',  3.60, 4.00, 3.70, '#A0522D',  'Continuum'),
    ('1600 A',  3.60, 4.00, 3.70, '#CD853F',  'UV cont.'),
    ('1216 A',  4.00, 4.48, 4.30, '#FF8C00',  'H I Ly-a'),
    ('1550 A',  4.78, 5.40, 5.00, '#FF4500',  'C IV'),
    ('171 A',   5.20, 6.30, 5.95, '#1E90FF',  'Fe IX/X'),
    ('195 A',   5.70, 7.41, 6.18, '#32CD32',  'Fe XII/XXIV'),
    ('284 A',   6.10, 6.60, 6.30, '#9400D3',  'Fe XV'),
]

fig, ax = plt.subplots(figsize=(14, 6))

y_positions = np.arange(len(trace_channels))
bar_height = 0.6

for i, (name, log_tmin, log_tmax, log_tpeak, color, ion) in enumerate(
    trace_channels
):
    # Temperature range bar.
    ax.barh(
        i, log_tmax - log_tmin, left=log_tmin,
        height=bar_height, color=color, alpha=0.7,
        edgecolor='black', linewidth=0.8
    )
    # Peak temperature marker.
    ax.plot(log_tpeak, i, 'k*', markersize=12, zorder=5)

    # Label.
    ax.text(
        log_tmax + 0.05, i,
        f'{name}  ({ion})',
        va='center', fontsize=9, fontweight='bold'
    )

# Solar atmosphere regions.
regions = [
    (3.5, 4.0, 'Photosphere', '#FFF8DC'),
    (4.0, 4.5, 'Chromosphere', '#FFEFD5'),
    (4.5, 5.5, 'Transition\
Region', '#FFE4E1'),
    (5.5, 7.5, 'Corona', '#E6E6FA'),
]

for log_t1, log_t2, label, fc in regions:
    ax.axvspan(log_t1, log_t2, alpha=0.15, color=fc, zorder=0)
    ax.text(
        (log_t1 + log_t2) / 2, len(trace_channels) - 0.3,
        label, ha='center', va='bottom', fontsize=8,
        style='italic', color='gray'
    )

ax.set_yticks(y_positions)
ax.set_yticklabels([ch[0] for ch in trace_channels])
ax.set_xlabel('log T (K)')
ax.set_xlim(3.3, 7.8)
ax.set_ylim(-0.5, len(trace_channels) + 0.3)
ax.set_title('Part 2(a): TRACE Channel Temperature Coverage')
ax.grid(True, axis='x', alpha=0.3)

# Legend for peak marker.
ax.plot([], [], 'k*', markersize=10, label='Peak response')
ax.legend(loc='lower right')

plt.tight_layout()
plt.show()

# Print temperature ranges.
print("\
TRACE Channel Temperature Coverage:")
print(f"{'Channel':>8s}  {'T_min (K)':>12s}  {'T_max (K)':>12s}  {'T_peak (K)':>12s}")
for name, log_tmin, log_tmax, log_tpeak, _, ion in trace_channels:
    print(
        f"{name:>8s}  {10**log_tmin:>12.0f}  {10**log_tmax:>12.0f}  "
        f"{10**log_tpeak:>12.0f}  ({ion})"
    )

### (b) TRACE vs EIT 온도 커버리지 비교 / TRACE vs EIT Temperature Coverage Comparison

EIT는 4개 EUV 밴드(171/195/284/304 A)만 제공합니다. TRACE는 304 A (He II) 채널이 없지만, UV/WL 채널을 추가하여 광구에서 코로나까지 연속적인 온도 커버리지를 제공합니다.

EIT provides only 4 EUV bands (171/195/284/304 A). TRACE lacks the 304 A (He II) channel but adds UV/WL channels, providing continuous temperature coverage from photosphere to corona.

In [ ]:
# =============================================================================
# Part 2(b): TRACE vs EIT temperature coverage comparison
# =============================================================================

# EIT channels: (name, log_T_min, log_T_max, log_T_peak, color, ion).
eit_channels = [
    ('304 A',  4.70, 5.00, 4.85, '#FF6347', 'He II'),
    ('171 A',  5.20, 6.30, 5.95, '#1E90FF', 'Fe IX/X'),
    ('195 A',  5.70, 6.50, 6.18, '#32CD32', 'Fe XII'),
    ('284 A',  6.10, 6.60, 6.30, '#9400D3', 'Fe XV'),
]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Top: EIT channels.
ax = axes[0]
for i, (name, log_tmin, log_tmax, log_tpeak, color, ion) in enumerate(
    eit_channels
):
    ax.barh(
        i, log_tmax - log_tmin, left=log_tmin,
        height=0.6, color=color, alpha=0.7,
        edgecolor='black', linewidth=0.8
    )
    ax.plot(log_tpeak, i, 'k*', markersize=12, zorder=5)
    ax.text(
        log_tmax + 0.05, i,
        f'{name} ({ion})', va='center', fontsize=9, fontweight='bold'
    )

ax.set_yticks(range(len(eit_channels)))
ax.set_yticklabels([ch[0] for ch in eit_channels])
ax.set_ylim(-0.5, len(eit_channels) + 0.3)
ax.set_title('EIT: 4 EUV Channels Only')
ax.grid(True, axis='x', alpha=0.3)

# Highlight gap: no photosphere/chromosphere coverage.
ax.axvspan(3.3, 4.5, alpha=0.1, color='red', zorder=0)
ax.text(
    3.9, len(eit_channels) - 0.3,
    'No coverage\
(photosphere-\
chromosphere)',
    ha='center', va='bottom', fontsize=8, color='red', style='italic'
)

# Bottom: TRACE channels.
ax2 = axes[1]
for i, (name, log_tmin, log_tmax, log_tpeak, color, ion) in enumerate(
    trace_channels
):
    ax2.barh(
        i, log_tmax - log_tmin, left=log_tmin,
        height=0.6, color=color, alpha=0.7,
        edgecolor='black', linewidth=0.8
    )
    ax2.plot(log_tpeak, i, 'k*', markersize=12, zorder=5)
    ax2.text(
        log_tmax + 0.05, i,
        f'{name} ({ion})', va='center', fontsize=9, fontweight='bold'
    )

ax2.set_yticks(range(len(trace_channels)))
ax2.set_yticklabels([ch[0] for ch in trace_channels])
ax2.set_ylim(-0.5, len(trace_channels) + 0.3)
ax2.set_xlabel('log T (K)')
ax2.set_title('TRACE: 8 Channels (EUV + UV + WL)')
ax2.set_xlim(3.3, 7.8)
ax2.grid(True, axis='x', alpha=0.3)

# Highlight gap: no 304 A coverage.
ax2.axvspan(4.5, 4.9, alpha=0.1, color='red', zorder=0)
ax2.text(
    4.7, len(trace_channels) - 0.3,
    'No 304 A\
(He II)',
    ha='center', va='bottom', fontsize=8, color='red', style='italic'
)

# Add region labels to bottom panel.
for log_t1, log_t2, label, fc in regions:
    ax2.axvspan(log_t1, log_t2, alpha=0.08, color=fc, zorder=0)

fig.suptitle(
    'Part 2(b): Temperature Coverage — EIT (4 bands) vs TRACE (8 bands)',
    fontsize=13, y=1.02
)
plt.tight_layout()
plt.show()

print("\
Key differences:")
print("  - EIT has 304 A (He II, ~7x10^4 K) — TRACE does NOT")
print("  - TRACE adds UV channels: Ly-alpha (1216), C IV (1550), UV cont. (1600, 1700)")
print("  - TRACE adds white light channel for photospheric context")
print("  - TRACE provides continuous coverage: photosphere -> TR -> corona")

---
## Part 3: 코로나 루프 분해능 시뮬레이션 / Coronal Loop Resolution Simulation

TRACE의 0.5" 분해능이 코로나 물리학에 미친 과학적 영향을 시뮬레이션합니다. 코로나 루프의 내부 가닥(strand) 구조가 EIT와 TRACE에서 어떻게 보이는지 비교합니다.

We simulate the scientific impact of TRACE's 0.5" resolution on coronal physics. We compare how the internal strand structure of coronal loops appears at EIT vs TRACE resolution.

### (a) 코로나 루프 시뮬레이션 / Coronal Loop Simulation
- 반원형 루프: ~50 Mm 길이, ~1 Mm 폭 / Semicircular loop: ~50 Mm length, ~1 Mm width
- 내부 가닥: ~200 km 폭, 루프당 5개 / Internal strands: ~200 km width, 5 per loop
- EIT (2.6")에서는 가닥 분해 불가 / Strands unresolved by EIT (2.6")
- TRACE (0.5")에서는 부분적으로 분해 가능 / Strands partially resolved by TRACE (0.5")

In [ ]:
# =============================================================================
# Part 3(a): Coronal loop strand simulation — EIT vs TRACE resolution
# =============================================================================

# Physical parameters.
km_per_arcsec = 725.0
loop_radius_km = 25000.0       # ~25 Mm radius -> ~50 Mm semicircle length.
strand_width_km = 200.0        # Individual strand width.
n_strands = 5                  # Number of strands per loop.
loop_total_width_km = 1000.0   # Total loop bundle width ~1 Mm.

# Convert to arcsec.
loop_radius_arcsec = loop_radius_km / km_per_arcsec
strand_width_arcsec = strand_width_km / km_per_arcsec
loop_width_arcsec = loop_total_width_km / km_per_arcsec

print(f"Loop radius: {loop_radius_arcsec:.1f} arcsec ({loop_radius_km/1000:.0f} Mm)")
print(f"Strand width: {strand_width_arcsec:.2f} arcsec ({strand_width_km:.0f} km)")
print(f"Loop bundle width: {loop_width_arcsec:.2f} arcsec ({loop_total_width_km:.0f} km)")
print(f"\
EIT pixel (2.6\") = {2.6 * km_per_arcsec:.0f} km -> strands UNRESOLVED")
print(f"TRACE pixel (0.5\") = {0.5 * km_per_arcsec:.0f} km -> strands PARTIALLY resolved")

# Create synthetic loop image on a high-resolution grid.
# Grid: 100" x 50" at 0.05"/pixel (supersampled).
grid_sampling = 0.05  # arcsec/pixel.
x_range = np.arange(-50, 50 + grid_sampling, grid_sampling)
y_range = np.arange(-5, 50 + grid_sampling, grid_sampling)
xx, yy = np.meshgrid(x_range, y_range)

# Semicircular loop: center at (0, 0), strands offset perpendicular.
image_truth = np.zeros_like(xx)

# Strand offsets from loop center (perpendicular to loop direction).
strand_offsets = np.linspace(
    -loop_width_arcsec / 2, loop_width_arcsec / 2, n_strands
)

for offset in strand_offsets:
    # Distance from semicircle of radius R + offset.
    r = np.sqrt(xx**2 + yy**2)
    dist_from_loop = np.abs(r - (loop_radius_arcsec + offset))

    # Gaussian strand profile.
    sigma_strand = strand_width_arcsec / (2 * np.sqrt(2 * np.log(2)))
    strand = np.exp(-0.5 * (dist_from_loop / sigma_strand) ** 2)

    # Only upper semicircle (y >= 0).
    strand[yy < 0] = 0
    image_truth += strand

# Normalize.
image_truth /= image_truth.max()

# Simulate EIT and TRACE observations by convolving with their PSFs.
# PSF FWHM in pixels of our supersampled grid.
eit_fwhm_pix = eit_pixel_arcsec / grid_sampling
trace_fwhm_pix = trace_pixel_arcsec / grid_sampling

eit_sigma_pix = eit_fwhm_pix / (2 * np.sqrt(2 * np.log(2)))
trace_sigma_pix = trace_fwhm_pix / (2 * np.sqrt(2 * np.log(2)))

image_eit = gaussian_filter(image_truth, sigma=eit_sigma_pix)
image_trace = gaussian_filter(image_truth, sigma=trace_sigma_pix)

# Plot.
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

extent = [x_range[0], x_range[-1], y_range[0], y_range[-1]]

for ax, img, title in zip(
    axes,
    [image_truth, image_trace, image_eit],
    ['Truth (5 strands, 200 km each)',
     'TRACE (0.5"/pix)',
     'EIT (2.6"/pix)']
):
    im = ax.imshow(
        img, extent=extent, origin='lower', cmap='hot',
        vmin=0, vmax=1, aspect='equal'
    )
    ax.set_title(title)
    ax.set_xlabel('Arcsec')
    ax.set_ylabel('Arcsec')
    ax.set_xlim(-45, 45)
    ax.set_ylim(-2, 42)
    fig.colorbar(im, ax=ax, shrink=0.7)

fig.suptitle(
    'Part 3(a): Coronal Loop — 5 Strands at Different Resolutions',
    fontsize=13, y=1.02
)
plt.tight_layout()
plt.show()

### (b) 다중 온도 가닥 모델 / Multi-thermal Strand Model

각 가닥이 약간씩 다른 온도(0.8, 0.9, 1.0, 1.1, 1.2 MK)를 가지며, 171 A와 195 A 채널에서 서로 다른 가닥이 지배적으로 보입니다. 이것이 다중 온도 루프 모델의 관측적 근거입니다.

Each strand has a slightly different temperature (0.8, 0.9, 1.0, 1.1, 1.2 MK), and different strands dominate in the 171 A vs 195 A channels. This is the observational basis for the multi-thermal loop model.

In [ ]:
# =============================================================================
# Part 3(b): Multi-thermal strand model — 171 vs 195 A appearance
# =============================================================================

def temperature_response_171(log_t: float) -> float:
    """Approximate TRACE 171 A temperature response function.

    Peaked at log T ~ 5.95 (~ 0.9 MK), dominated by Fe IX/X.

    Args:
        log_t: Log10 of temperature in Kelvin.

    Returns:
        Normalized response value.
    """
    return np.exp(-0.5 * ((log_t - 5.95) / 0.15) ** 2)


def temperature_response_195(log_t: float) -> float:
    """Approximate TRACE 195 A temperature response function.

    Peaked at log T ~ 6.18 (~ 1.5 MK), dominated by Fe XII.

    Args:
        log_t: Log10 of temperature in Kelvin.

    Returns:
        Normalized response value.
    """
    return np.exp(-0.5 * ((log_t - 6.18) / 0.15) ** 2)


# Strand temperatures.
strand_temps_mk = [0.8, 0.9, 1.0, 1.1, 1.2]  # MK
strand_log_t = [np.log10(t * 1e6) for t in strand_temps_mk]

# Response of each strand in each channel.
resp_171 = [temperature_response_171(lt) for lt in strand_log_t]
resp_195 = [temperature_response_195(lt) for lt in strand_log_t]

print("Strand temperature response:")
print(f"{'Strand':>8s}  {'T (MK)':>8s}  {'log T':>8s}  {'171 A':>8s}  {'195 A':>8s}")
for i, (t, lt, r1, r2) in enumerate(
    zip(strand_temps_mk, strand_log_t, resp_171, resp_195)
):
    dominant = '171' if r1 > r2 else '195'
    print(f"{'#' + str(i+1):>8s}  {t:>8.1f}  {lt:>8.3f}  {r1:>8.3f}  {r2:>8.3f}  -> {dominant} A")

# Create cross-sectional profiles for each channel.
# Use a 1D slice across the loop strands.
cross_section_arcsec = np.linspace(-1.5, 1.5, 1000)

profile_171 = np.zeros_like(cross_section_arcsec)
profile_195 = np.zeros_like(cross_section_arcsec)

for offset, r1, r2 in zip(strand_offsets, resp_171, resp_195):
    sigma_s = strand_width_arcsec / (2 * np.sqrt(2 * np.log(2)))
    strand_profile = np.exp(-0.5 * ((cross_section_arcsec - offset) / sigma_s) ** 2)
    profile_171 += r1 * strand_profile
    profile_195 += r2 * strand_profile

# Normalize.
max_val = max(profile_171.max(), profile_195.max())
profile_171 /= max_val
profile_195 /= max_val

# Apply TRACE PSF.
trace_sigma_cs = (trace_pixel_arcsec / (2 * np.sqrt(2 * np.log(2)))) / \
    (cross_section_arcsec[1] - cross_section_arcsec[0])
profile_171_trace = gaussian_filter(profile_171, sigma=trace_sigma_cs)
profile_195_trace = gaussian_filter(profile_195, sigma=trace_sigma_cs)

# Apply EIT PSF.
eit_sigma_cs = (eit_pixel_arcsec / (2 * np.sqrt(2 * np.log(2)))) / \
    (cross_section_arcsec[1] - cross_section_arcsec[0])
profile_171_eit = gaussian_filter(profile_171, sigma=eit_sigma_cs)
profile_195_eit = gaussian_filter(profile_195, sigma=eit_sigma_cs)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Truth.
axes[0].plot(cross_section_arcsec, profile_171, 'b-', lw=2, label='171 A')
axes[0].plot(cross_section_arcsec, profile_195, 'g-', lw=2, label='195 A')
for i, offset in enumerate(strand_offsets):
    axes[0].axvline(offset, color='gray', ls=':', alpha=0.4)
    axes[0].text(
        offset, 1.05, f'{strand_temps_mk[i]} MK',
        ha='center', fontsize=7, rotation=45
    )
axes[0].set_title('Truth: Individual Strands')
axes[0].set_xlabel('Cross-section (arcsec)')
axes[0].set_ylabel('Normalized Intensity')
axes[0].legend()
axes[0].set_ylim(0, 1.15)

# TRACE resolution.
axes[1].plot(cross_section_arcsec, profile_171_trace, 'b-', lw=2, label='171 A')
axes[1].plot(cross_section_arcsec, profile_195_trace, 'g-', lw=2, label='195 A')
axes[1].set_title('TRACE Resolution (0.5"/pix)')
axes[1].set_xlabel('Cross-section (arcsec)')
axes[1].set_ylabel('Normalized Intensity')
axes[1].legend()
axes[1].set_ylim(0, 1.15)

# EIT resolution.
axes[2].plot(cross_section_arcsec, profile_171_eit, 'b-', lw=2, label='171 A')
axes[2].plot(cross_section_arcsec, profile_195_eit, 'g-', lw=2, label='195 A')
axes[2].set_title('EIT Resolution (2.6"/pix)')
axes[2].set_xlabel('Cross-section (arcsec)')
axes[2].set_ylabel('Normalized Intensity')
axes[2].legend()
axes[2].set_ylim(0, 1.15)

for ax in axes:
    ax.grid(True, alpha=0.3)

fig.suptitle(
    'Part 3(b): Multi-thermal Loop — 171 vs 195 A Cross-section',
    fontsize=13, y=1.02
)
plt.tight_layout()
plt.show()

print("\
Key insight: At TRACE resolution, the two channels show different")
print("cross-sectional profiles, revealing multi-thermal strand structure.")
print("At EIT resolution, both channels show a single unresolved blob.")

---
## Part 4: 데이터 볼륨 비교 / Data Volume Comparison

EUV 영상 기기의 데이터 처리 능력 진화를 비교합니다. EIT에서 TRACE, AIA로의 발전은 텔레메트리와 온보드 저장 기술의 혁신을 반영합니다.

We compare the evolution of data handling capabilities across EUV imagers. The progression from EIT to TRACE to AIA reflects innovations in telemetry and onboard storage.

### 주요 비교 항목 / Key Comparison Items
- 일일 데이터 볼륨 / Daily data volume
- 영상 케이던스 / Image cadence
- 미션 수명 동안의 누적 데이터 / Cumulative data over mission lifetime

In [ ]:
# =============================================================================
# Part 4(a-b): Daily data volume and cadence comparison
# =============================================================================

# Instrument data parameters.
instruments_data = {
    'EIT': {
        'daily_mb': 10,        # ~1 kbit/s -> ~10 MB/day (compressed)
        'cadence_s': 720,      # 12 min per full-disk image
        'mission_start': 1996,
        'mission_end': 2010,   # SOHO still operates, but EIT largely replaced
        'color': 'red',
        'fov_type': 'Full disk',
        'n_channels': 4,
    },
    'TRACE': {
        'daily_mb': 700,       # ~700 MB/day
        'cadence_s': 30,       # ~30s per image (partial FOV)
        'mission_start': 1998,
        'mission_end': 2010,
        'color': 'blue',
        'fov_type': 'Partial (8.5\')',
        'n_channels': 8,
    },
    'AIA': {
        'daily_mb': 1_500_000, # ~1.5 TB/day
        'cadence_s': 12,       # 12s per image, all 10 channels
        'mission_start': 2010,
        'mission_end': 2025,   # Still operating
        'color': 'green',
        'fov_type': 'Full disk',
        'n_channels': 10,
    },
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) Daily data volume — bar chart.
names = list(instruments_data.keys())
daily_vols = [instruments_data[n]['daily_mb'] for n in names]
colors_list = [instruments_data[n]['color'] for n in names]

bars = axes[0].bar(names, daily_vols, color=colors_list, edgecolor='black',
                   alpha=0.8)
axes[0].set_yscale('log')
axes[0].set_ylabel('Daily Data Volume (MB/day)')
axes[0].set_title('(a) Daily Data Volume')
axes[0].grid(True, axis='y', alpha=0.3)

# Add value labels.
for bar, vol in zip(bars, daily_vols):
    if vol >= 1000:
        label = f'{vol/1000:.0f} GB' if vol < 1e6 else f'{vol/1e6:.1f} TB'
    else:
        label = f'{vol} MB'
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.3,
        label, ha='center', fontsize=10, fontweight='bold'
    )

# (b) Image cadence — bar chart.
cadences = [instruments_data[n]['cadence_s'] for n in names]
bars2 = axes[1].bar(names, cadences, color=colors_list, edgecolor='black',
                    alpha=0.8)
axes[1].set_yscale('log')
axes[1].set_ylabel('Image Cadence (seconds)')
axes[1].set_title('(b) Image Cadence')
axes[1].grid(True, axis='y', alpha=0.3)

# Add value labels.
for bar, cad in zip(bars2, cadences):
    label = f'{cad}s' if cad < 60 else f'{cad/60:.0f} min'
    axes[1].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.3,
        label, ha='center', fontsize=10, fontweight='bold'
    )

# Add FOV type annotations.
for bar, name in zip(bars2, names):
    fov_type = instruments_data[name]['fov_type']
    axes[1].text(
        bar.get_x() + bar.get_width() / 2, 5,
        fov_type, ha='center', fontsize=7, style='italic',
        color='white', fontweight='bold'
    )

# (c) Cumulative data volume over mission lifetime.
ax3 = axes[2]
for name, data in instruments_data.items():
    years = np.arange(data['mission_start'], data['mission_end'] + 1)
    daily_gb = data['daily_mb'] / 1000.0  # Convert to GB.
    cumulative_tb = np.cumsum(np.ones(len(years)) * daily_gb * 365) / 1000.0
    ax3.plot(years, cumulative_tb, '-o', color=data['color'],
             lw=2, markersize=3, label=name)

ax3.set_yscale('log')
ax3.set_xlabel('Year')
ax3.set_ylabel('Cumulative Data (TB)')
ax3.set_title('(c) Cumulative Data Over Mission Lifetime')
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.set_xlim(1994, 2027)

fig.suptitle(
    'Part 4: Data Volume & Cadence Comparison — EIT / TRACE / AIA',
    fontsize=13, y=1.02
)
plt.tight_layout()
plt.show()

# Print summary table.
print("\
Data Volume Comparison:")
print(f"{'Instrument':>12s}  {'Daily Volume':>14s}  {'Cadence':>10s}  "
      f"{'Channels':>10s}  {'FOV':>15s}")
for name, data in instruments_data.items():
    vol = data['daily_mb']
    if vol >= 1e6:
        vol_str = f"{vol/1e6:.1f} TB"
    elif vol >= 1000:
        vol_str = f"{vol/1000:.0f} GB"
    else:
        vol_str = f"{vol} MB"
    cad = data['cadence_s']
    cad_str = f"{cad}s" if cad < 60 else f"{cad/60:.0f} min"
    print(
        f"{name:>12s}  {vol_str:>14s}  {cad_str:>10s}  "
        f"{data['n_channels']:>10d}  {data['fov_type']:>15s}"
    )

---
## Part 5: EUV 영상 진화 / EUV Imaging Evolution

NIXT (1989)에서 EUI-HRI (2020)까지, EUV 태양 영상 기기의 30년 진화를 종합적으로 비교합니다.

Comprehensive comparison of the 30-year evolution of EUV solar imagers, from NIXT (1989) to EUI-HRI (2020).

| | NIXT | EIT | TRACE | EUVI | AIA | EUI-HRI |
|---|---|---|---|---|---|---|
| Year | 1989 | 1995 | 1998 | 2006 | 2010 | 2020 |
| Aperture | 25 cm | 12 cm | 30 cm | 10 cm | 20 cm | 17 cm |
| Pixel | ~0.75" | 2.6" | 0.5" | 1.6" | 0.6" | 0.5" |
| FOV | 30' | 45' | 8.5' | 54' | 41' | 17' |
| Channels | 1 | 4 | 8 | 4 | 10 | 2 |
| Cadence | snapshot | 12 min | 30s | 2.5 min | 12s | 2-5s |
| Orbit | rocket | L1 | LEO | L4/L5 | GEO | heliocentric |

In [ ]:
# =============================================================================
# Part 5: EUV Imaging Evolution — comprehensive comparison
# =============================================================================

# Instrument database.
euv_imagers = {
    'NIXT':    {'year': 1989, 'aperture_cm': 25, 'pixel_arcsec': 0.75,
                'fov_arcmin': 30,  'channels': 1,  'cadence_s': np.nan,
                'orbit': 'Rocket'},
    'EIT':     {'year': 1995, 'aperture_cm': 12, 'pixel_arcsec': 2.6,
                'fov_arcmin': 45,  'channels': 4,  'cadence_s': 720,
                'orbit': 'L1'},
    'TRACE':   {'year': 1998, 'aperture_cm': 30, 'pixel_arcsec': 0.5,
                'fov_arcmin': 8.5, 'channels': 8,  'cadence_s': 30,
                'orbit': 'LEO'},
    'EUVI':    {'year': 2006, 'aperture_cm': 10, 'pixel_arcsec': 1.6,
                'fov_arcmin': 54,  'channels': 4,  'cadence_s': 150,
                'orbit': 'L4/L5'},
    'AIA':     {'year': 2010, 'aperture_cm': 20, 'pixel_arcsec': 0.6,
                'fov_arcmin': 41,  'channels': 10, 'cadence_s': 12,
                'orbit': 'GEO'},
    'EUI-HRI': {'year': 2020, 'aperture_cm': 17, 'pixel_arcsec': 0.5,
                'fov_arcmin': 17,  'channels': 2,  'cadence_s': 3,
                'orbit': 'Helio'},
}

names_euv = list(euv_imagers.keys())
years = [euv_imagers[n]['year'] for n in names_euv]
pixels = [euv_imagers[n]['pixel_arcsec'] for n in names_euv]
cadences_euv = [euv_imagers[n]['cadence_s'] for n in names_euv]
apertures = [euv_imagers[n]['aperture_cm'] for n in names_euv]
fovs_euv = [euv_imagers[n]['fov_arcmin'] for n in names_euv]
channels = [euv_imagers[n]['channels'] for n in names_euv]

# Color scheme: highlight TRACE.
colors_euv = ['#999999', '#CC4444', '#2266CC', '#999999', '#44AA44', '#FF8800']

fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

# (1) Pixel resolution over time.
ax1 = fig.add_subplot(gs[0, 0])
ax1.bar(names_euv, pixels, color=colors_euv, edgecolor='black', alpha=0.8)
ax1.set_ylabel('Pixel Scale (arcsec)')
ax1.set_title('Spatial Resolution')
ax1.grid(True, axis='y', alpha=0.3)
for i, (n, p) in enumerate(zip(names_euv, pixels)):
    ax1.text(i, p + 0.08, f'{p}"', ha='center', fontsize=8, fontweight='bold')
ax1.tick_params(axis='x', rotation=30)

# (2) Cadence over time.
ax2 = fig.add_subplot(gs[0, 1])
valid_cad = [(n, c, col) for n, c, col in
             zip(names_euv, cadences_euv, colors_euv) if not np.isnan(c)]
ax2.bar(
    [v[0] for v in valid_cad],
    [v[1] for v in valid_cad],
    color=[v[2] for v in valid_cad],
    edgecolor='black', alpha=0.8
)
ax2.set_yscale('log')
ax2.set_ylabel('Cadence (seconds)')
ax2.set_title('Temporal Cadence')
ax2.grid(True, axis='y', alpha=0.3)
for n, c, _ in valid_cad:
    label = f'{c:.0f}s' if c < 60 else f'{c/60:.0f} min'
    idx = [v[0] for v in valid_cad].index(n)
    ax2.text(idx, c * 1.3, label, ha='center', fontsize=8, fontweight='bold')
ax2.tick_params(axis='x', rotation=30)

# (3) Number of channels.
ax3 = fig.add_subplot(gs[0, 2])
ax3.bar(names_euv, channels, color=colors_euv, edgecolor='black', alpha=0.8)
ax3.set_ylabel('Number of Channels')
ax3.set_title('Spectral Coverage')
ax3.grid(True, axis='y', alpha=0.3)
for i, (n, ch) in enumerate(zip(names_euv, channels)):
    ax3.text(i, ch + 0.2, str(ch), ha='center', fontsize=10, fontweight='bold')
ax3.tick_params(axis='x', rotation=30)

# (4) FOV.
ax4 = fig.add_subplot(gs[1, 0])
ax4.bar(names_euv, fovs_euv, color=colors_euv, edgecolor='black', alpha=0.8)
ax4.set_ylabel('FOV (arcmin)')
ax4.set_title('Field of View')
ax4.grid(True, axis='y', alpha=0.3)
ax4.axhline(32, color='orange', ls='--', alpha=0.5, label='Solar diameter')
ax4.legend(fontsize=8)
for i, (n, f) in enumerate(zip(names_euv, fovs_euv)):
    ax4.text(i, f + 1, f"{f}'", ha='center', fontsize=8, fontweight='bold')
ax4.tick_params(axis='x', rotation=30)

# (5) Aperture.
ax5 = fig.add_subplot(gs[1, 1])
ax5.bar(names_euv, apertures, color=colors_euv, edgecolor='black', alpha=0.8)
ax5.set_ylabel('Aperture (cm)')
ax5.set_title('Primary Aperture')
ax5.grid(True, axis='y', alpha=0.3)
for i, (n, a) in enumerate(zip(names_euv, apertures)):
    ax5.text(i, a + 0.5, f'{a} cm', ha='center', fontsize=8, fontweight='bold')
ax5.tick_params(axis='x', rotation=30)

# (6) Timeline.
ax6 = fig.add_subplot(gs[1, 2])
ax6.scatter(years, range(len(years)), s=200, c=colors_euv,
            edgecolors='black', zorder=5)
for i, (n, y) in enumerate(zip(names_euv, years)):
    ax6.annotate(
        f'{n} ({y})', (y, i),
        textcoords='offset points', xytext=(15, 0),
        fontsize=9, va='center', fontweight='bold',
        color=colors_euv[i]
    )
ax6.set_xlabel('Year')
ax6.set_title('Launch Timeline')
ax6.set_xlim(1985, 2025)
ax6.set_yticks([])
ax6.grid(True, axis='x', alpha=0.3)

fig.suptitle(
    'Part 5: EUV Solar Imager Evolution (1989-2020)',
    fontsize=14, y=1.01
)
plt.show()

---
## Summary / 요약

TRACE는 EIT와 AIA 사이의 가교 역할을 하며, 고분해능 코로나 물리학의 새 시대를 열었습니다.

TRACE bridged the gap between EIT and AIA, opening a new era of high-resolution coronal physics.

### TRACE의 핵심 기여 / Key Contributions of TRACE

| Aspect | EIT (1995) | TRACE (1998) | AIA (2010) |
|--------|-----------|-------------|------------|
| Resolution | 2.6"/pix | **0.5"/pix** | 0.6"/pix |
| FOV | 45' (full disk) | 8.5' (partial) | 41' (full disk) |
| Cadence | 12 min | **30s** | 12s |
| Channels | 4 EUV | 3 EUV + 4 UV + WL | 7 EUV + 2 UV + WL |
| Data rate | 10 MB/day | **700 MB/day** | 1.5 TB/day |
| Multilayers | Mo/Si | **Mo2C/Si** | Mo/Si (optimized) |

### TRACE가 가능하게 한 과학 / Science Enabled by TRACE

1. **코로나 루프 미세 구조 발견** — 루프가 분해되지 않는 가닥 다발임을 최초 관측으로 입증
   Discovery of coronal loop fine structure — first observational evidence that loops are bundles of unresolved strands

2. **다중 온도 루프 모델** — 같은 루프가 171/195 A에서 다르게 보임 → 가닥별 온도 차이
   Multi-thermal loop model — same loop appears different in 171 vs 195 A, revealing strand-by-strand temperature variation

3. **코로나 가열 제약 조건** — 높은 시간/공간 분해능으로 가열 메커니즘에 대한 새로운 제약 제공
   Coronal heating constraints — high temporal/spatial resolution provided new constraints on heating mechanisms

4. **광구-코로나 연결** — UV/WL 채널로 광구 자기장과 코로나 구조의 연결 추적 가능
   Photosphere-corona connection — UV/WL channels enabled tracking connections between photospheric fields and coronal structures

---

**Paper**: Handy, B.N., et al. (1999). "The Transition Region and Coronal Explorer." *Solar Physics*, 187, 229-260. [DOI: 10.1023/A:1005166902804]